# NB-07 — T5 OrderBatch + XLSX workbook walkthrough

Ships alongside T5 phases P1 (#1749) and P2 (#1752) of #1719.

## What this notebook covers

- Why T5 uses a file-drop instead of a broker API (Fidelity has no API)
- Constructing an `OrderTicket` with the security guards it enforces
- Composing an `OrderBatch` and understanding the SHA idempotency invariant
- Writing the paired **CSV + XLSX** artifact via `PaperOrderSink`
- Walking through each of the **6 XLSX sheets** so a reviewer knows what to look at

## What this notebook does NOT cover

- The paper trading engine (`SqlitePaperEngine`) — see **NB-08**
- The full end-to-end plan-to-fill flow — see **NB-09**
- Fidelity's actual position download format — see `docs/superpowers/specs/2026-08-03-fidelity-positions-csv-schema.md`

The full E2E test guide is at `docs/superpowers/specs/2026-08-03-t5-e2e-test-guide.md`.


## 1. Why T5 is file-drop + manual placement

Fidelity — the operator's actual broker — does not expose any API
that accepts a structured order upload. Every order must be filed
manually at Fidelity's regular order-entry UI.

That reframes T5 entirely. Instead of routing orders through a broker
adapter, T5 writes two disk artifacts:

- **CSV** — a portable table of the orders the operator will file
- **XLSX** — a human-review workbook with 6 sheets that give the
  reviewer everything they need to sanity-check the batch **before**
  they start typing orders into Fidelity's UI

Both files share the same batch SHA in the filename so the pair is
trivially matchable in a shared drive.


## 2. OrderTicket — the executable unit

Each ticket is a frozen dataclass with allowlist validation on
construction. Bad tickets never reach the writer.


In [ ]:
from decimal import Decimal
from openbb_techtrade.execution.order_sink import OrderTicket

t = OrderTicket(
    symbol="MSFT",
    action="Buy",
    quantity=Decimal("50"),
    order_type="Limit",
    limit_price=Decimal("400.00"),
    tif="Day",
    account_masked="***1234",
    notes="Take profit ladder step 1",
)
t

OrderTicket(symbol='MSFT', action='Buy', quantity=Decimal('50'), order_type='Limit', limit_price=Decimal('400.00'), tif='Day', account_masked='***1234', notes='Take profit ladder step 1')

### Guards enforced at construction

The constructor rejects many classes of bad input **loudly** —
no silent coercion. Below: three things it refuses.


In [ ]:
# Formula-injection guard — a note starting with `=` / `+` / `-` / `@` / TAB / CR
# is rejected because Excel would evaluate it as a formula on double-click.
try:
    OrderTicket(symbol="MSFT", action="Buy", quantity=Decimal("1"), notes="=cmd|/C calc")
except ValueError as e:
    print("Rejected notes: ", str(e)[:100], "...")

# Symbol allowlist — lower case, spaces, and specials are refused.
try:
    OrderTicket(symbol="msft!", action="Buy", quantity=Decimal("1"))
except ValueError as e:
    print("Rejected symbol:", str(e)[:100], "...")

# Quantity must be positive.
try:
    OrderTicket(symbol="MSFT", action="Buy", quantity=Decimal("0"))
except ValueError as e:
    print("Rejected qty:   ", str(e)[:100], "...")

Rejected notes:  OrderTicket.notes must not start with a CSV/Excel formula-injection character ['\t', '\r', '+', '-', ...
Rejected symbol: OrderTicket.symbol must match '^[A-Z][A-Z0-9./\\-]{0,15}$'; got 'msft!' ...
Rejected qty:    OrderTicket.quantity must be positive; got 0 ...


## 3. OrderBatch — a set of tickets + provenance

Compose several tickets into an `OrderBatch`. Optional P2 fields
(`pricing`, `pre_execution_positions`, `plan_context`) light up the
richer XLSX sheets when supplied.


In [ ]:
from openbb_techtrade.execution.order_sink import OrderBatch, PlanContext, VerdictGate

batch = OrderBatch(
    tickets=(
        OrderTicket(symbol="MSFT", action="Buy",  quantity=Decimal("50"),
                    order_type="Limit", limit_price=Decimal("400.00"), account_masked="***1234"),
        OrderTicket(symbol="AAPL", action="Buy",  quantity=Decimal("100"),
                    order_type="Limit", limit_price=Decimal("180.00"), account_masked="***1234"),
        OrderTicket(symbol="NVDA", action="Sell", quantity=Decimal("25"),
                    order_type="Limit", limit_price=Decimal("130.00"),
                    account_masked="***1234", notes="Take profit on rally"),
    ),
    plan_id="nb07-demo-2026",
    verdict_gate_pass=True,
    pricing={"MSFT": Decimal("398.00"), "AAPL": Decimal("179.50"), "NVDA": Decimal("128.00")},
    pre_execution_positions={"MSFT": Decimal("0.05"), "AAPL": Decimal("0.03"), "NVDA": Decimal("0.12")},
    plan_context=PlanContext(
        verdict_gates=(
            VerdictGate(name="max_single_position", threshold="0.10", actual="0.08", passed=True),
            VerdictGate(name="max_sector_exposure", threshold="0.30", actual="0.35", passed=False,
                        notes="Tech cluster exceeds — reviewer sign-off required"),
        ),
        generator_version="nb07-1.0",
        git_sha="0000000000",
    ),
)
print(f"tickets: {len(batch.tickets)}")
print(f"plan_id: {batch.plan_id}")
print(f"verdict_gate_pass: {batch.verdict_gate_pass}")
print(f"SHA (short): {batch.sha_short()}")
print(f"SHA (full):  {batch.sha256()}")

tickets: 3
plan_id: nb07-demo-2026
verdict_gate_pass: True
SHA (short): 6f368055
SHA (full):  6f368055d33ebb327cd95db249af3c636e5d112ef1cc8ae38dd4677d7f56bd7a


### SHA idempotency

`sha256()` is computed **only** over the canonical ticket sequence.
`plan_id`, `pricing`, and `plan_context` are excluded, so a re-priced
re-run of the same plan produces the same SHA and lands at the same
filename — safe for retry.


In [ ]:
same_tickets_different_pricing = OrderBatch(
    tickets=batch.tickets,
    plan_id="different-plan-id",
    pricing={"MSFT": Decimal("999")},  # completely different
)
print("Same tickets, different metadata → same SHA?",
      same_tickets_different_pricing.sha256() == batch.sha256())

# Different tickets → different SHA
one_more_ticket = OrderBatch(
    tickets=batch.tickets + (
        OrderTicket(symbol="GOOGL", action="Buy", quantity=Decimal("10"),
                    order_type="Market"),
    ),
)
print("Extra ticket → different SHA?",
      one_more_ticket.sha256() != batch.sha256())

Same tickets, different metadata → same SHA? True
Extra ticket → different SHA? True


## 4. Writing the paired CSV + XLSX

`PaperOrderSink` writes atomically to a caller-supplied output
directory. Write order is **XLSX first, CSV last** — so a mid-write
crash can never leave an uploadable CSV without its audit workbook.

For notebook execution we use a per-run temp dir so nothing lands in the
repo. In real usage this points at a per-operator export directory.


In [ ]:
import tempfile
from pathlib import Path
from openbb_techtrade.execution.order_sink import PaperOrderSink

out_dir = Path(tempfile.mkdtemp(prefix="nb07_"))
sink = PaperOrderSink(out_dir)
art = sink.write_batch(batch)

print(f"CSV:  {art.csv_path.name}")
print(f"XLSX: {art.xlsx_path.name}")
print(f"SHA:  {art.batch_sha256}")

CSV:  2026-08-03-6f368055.csv
XLSX: 2026-08-03-6f368055.xlsx
SHA:  6f368055d33ebb327cd95db249af3c636e5d112ef1cc8ae38dd4677d7f56bd7a


## 5. Inspecting the CSV — Fidelity Basket Trading format

Even though we don't upload this to Fidelity (they have no import
surface), we emit it in the shape a broker would expect: **no header
row**, 7 columns, one order per line.


In [ ]:
print(art.csv_path.read_text())

MSFT,Buy,50,Limit,400,Day,***1234
AAPL,Buy,100,Limit,180,Day,***1234
NVDA,Sell,25,Limit,130,Day,***1234



Column order: Symbol, Action, Quantity, Order Type, Limit Price, TIF, Account.
Empty Limit Price on Market orders is a bare comma. Never any leading
`=`/`+`/`-`/`@` — the formula-injection guard runs at ticket
construction so nothing bad can reach this file.


## 6. The 6-sheet XLSX workbook

The XLSX is what the operator actually reviews. Six sheets, in this
tab order:

1. **Orders** — executable rows, colored by side
2. **Batch Summary** — SHA + gross notional + verdict-gate rollup
3. **Plan Context** — one row per verdict gate, failed gates in red
4. **Deviation Analysis** — deviation % + horizontal bar chart
5. **Concentration** — pre + post weights + side-by-side pie charts
6. **Audit** — full SHA + provenance for post-facto reconciliation

Let's open the workbook and confirm each sheet.


In [ ]:
from openpyxl import load_workbook

wb = load_workbook(art.xlsx_path)
print(f"Sheet count: {len(wb.sheetnames)}")
print("Sheet tab order:")
for i, name in enumerate(wb.sheetnames, 1):
    print(f"  {i}. {name}")

Sheet count: 6
Sheet tab order:
  1. Orders
  2. Batch Summary
  3. Plan Context
  4. Deviation Analysis
  5. Concentration
  6. Audit


### Sheet 1 — Orders

In [ ]:
ws = wb["Orders"]
print(f"Rows: {ws.max_row} (1 header + {ws.max_row - 1} tickets)")
print(f"Columns: {ws.max_column}")

# Read the whole sheet as a plain list of lists.
for row in ws.iter_rows(values_only=True):
    print(row)

Rows: 4 (1 header + 3 tickets)
Columns: 8
('Symbol', 'Action', 'Quantity', 'Order Type', 'Limit Price', 'TIF', 'Account', 'Notes')
('MSFT', 'Buy', 50, 'Limit', 400, 'Day', '***1234', None)
('AAPL', 'Buy', 100, 'Limit', 180, 'Day', '***1234', None)
('NVDA', 'Sell', 25, 'Limit', 130, 'Day', '***1234', 'Take profit on rally')


Buys (MSFT, AAPL) have a green background; the sell (NVDA) has a
red background. The header row is frozen so scrolling large baskets
keeps it visible.


### Sheet 2 — Batch Summary

In [ ]:
ws = wb["Batch Summary"]
for row in ws.iter_rows(values_only=True):
    print(f"  {row[0]:<40} {row[1]}")

  Field                                    Value
  Batch SHA (short)                        6f368055
  Batch SHA (full)                         6f368055d33ebb327cd95db249af3c636e5d112ef1cc8ae38dd4677d7f56bd7a
  Plan ID                                  nb07-demo-2026
  Verdict gate                             PASS
  Total tickets                            3
  Buy tickets                              2
  Sell tickets                             1
  Gross notional (limit-priced only)       $41,250.00
  Generated at (UTC)                       2026-08-03T01:51:27+00:00


Notice **`Verdict gate: PASS`** — because we set
`verdict_gate_pass=True`. If it were False, the cell would be filled
with the same red highlight used on failed verdict gates in sheet 3.


### Sheet 3 — Plan Context

In [ ]:
ws = wb["Plan Context"]
for row in ws.iter_rows(values_only=True):
    print(row)

('Gate', 'Threshold', 'Actual', 'Passed', 'Notes')
('max_single_position', '0.10', '0.08', 'PASS', None)
('max_sector_exposure', '0.30', '0.35', 'FAIL', 'Tech cluster exceeds — reviewer sign-off required')


One row per gate. The `max_sector_exposure` row is filled red
because `passed=False`.


### Sheet 4 — Deviation Analysis + horizontal bar chart

In [ ]:
ws = wb["Deviation Analysis"]
print("Rows:")
for row in ws.iter_rows(values_only=True):
    print(row)

print(f"\nEmbedded charts: {len(ws._charts)}")
if ws._charts:
    chart = ws._charts[0]
    print(f"  Chart title: {chart.title.tx.rich.p[0].r[0].t if chart.title else 'n/a'}")
    print(f"  Chart type:  BarChart (horizontal)")


Rows:
('Symbol', 'Action', 'Quantity', 'Last Close', 'Limit Price', 'Deviation %', 'Notional')
('MSFT', 'Buy', 50, 398, 400, 0.503, 20000)
('AAPL', 'Buy', 100, 179.5, 180, 0.279, 18000)
('NVDA', 'Sell', 25, 128, 130, 1.562, 3250)

Embedded charts: 1
  Chart title: Deviation from last close (%)
  Chart type:  BarChart (horizontal)


Deviation % is `(limit − last_close) / last_close × 100`. Any
row with `|deviation| > 5%` gets an amber highlight — none of our
demo rows exceed 5%, so no highlight.


### Sheet 5 — Concentration + side-by-side pie charts

In [ ]:
ws = wb["Concentration"]
print("Rows:")
for row in ws.iter_rows(values_only=True):
    print(row)

print(f"\nEmbedded charts: {len(ws._charts)}")

Rows:
('Symbol', 'Pre-exec weight', 'Batch delta', 'Post-exec weight')
('AAPL', 0.03, 0.4364, 0.4664)
('MSFT', 0.05, 0.4848, 0.5348)
('NVDA', 0.12, -0.0788, 0.0412)

Embedded charts: 2


Rows are alphabetized (AAPL, MSFT, NVDA). Any symbol with
`|post-exec weight| > 10%` gets a red highlight — NVDA is at 12% pre-exec
so its row is flagged. Two pie charts render side-by-side: **Pre-execution**
on the left, **Post-execution** on the right, with matching category order.


### Sheet 6 — Audit

In [ ]:
ws = wb["Audit"]
for row in ws.iter_rows(values_only=True):
    print(f"  {row[0]:<30} {row[1]}")

  Field                          Value
  Batch SHA (full)               6f368055d33ebb327cd95db249af3c636e5d112ef1cc8ae38dd4677d7f56bd7a
  Batch SHA (short)              6f368055
  Plan ID                        nb07-demo-2026
  Verdict gate                   PASS
  Generator version              nb07-1.0
  Git SHA                        0000000000
  Generated at (UTC)             2026-08-03T01:51:27+00:00
  Written at (UTC)               2026-08-03T01:51:27+00:00
  Number of tickets              3
  Notes                          Written by openbb_techtrade.execution.order_sink._write_xlsx


Audit sheet is the post-facto reconciliation surface — carries
the full 64-char SHA plus the plan / generator / git provenance so a
reviewer months later can trace an executed batch back to the exact
code + plan version that produced it.


## 7. What's next

- **NB-08** — `08-t5-paper-trading-engine-walkthrough.ipynb` — how the
  paper engine records the fills the operator captures back from
  Fidelity
- **NB-09** — `09-t5-end-to-end-plan-to-fills.ipynb` — the full flow
  from validated plan through XLSX artifact through recorded fills

For the full E2E test guide (QA + automation scenarios), see
`docs/superpowers/specs/2026-08-03-t5-e2e-test-guide.md`.
